## Diagnósticos Clínicos MIMIC-III — DIAGNOSES_ICD

El conjunto de datos **DIAGNOSES_ICD** forma parte de la base de datos clínica MIMIC-III (Medical Information Mart for Intensive Care), desarrollada por el MIT Lab for Computational Physiology. Contiene los diagnósticos registrados en el sistema CIE-9-CM (ICD-9-CM) para cada hospitalización de pacientes en unidades de cuidados intensivos del Beth Israel Deaconess Medical Center (Boston, MA).

**Dataset:** [MIMIC-III Clinical Database Demo v1.4 — PhysioNet](https://physionet.org/content/mimiciii-demo/1.4/)

**Referencia:** Johnson, A., Pollard, T., & Mark, R. (2019). *MIMIC-III Clinical Database Demo* (version 1.4). PhysioNet. https://doi.org/10.13026/C2HM2Q

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

## 2. Carga y Selección de Datos

Se carga el archivo CSV y se muestran todas las columnas disponibles para el análisis.

| Columna (índice) | Nombre en el CSV | Descripción |
|------------------|------------------|-------------|
| 0 | <span style="color:red">`row_id`</span> | Identificador único de fila (índice interno, no aporta información clínica) |
| 1 | `subject_id` | Identificador único del paciente |
| 2 | `hadm_id` | Identificador único de la hospitalización (*Hospital Admission ID*) |
| 3 | `seq_num` | Número de secuencia del diagnóstico dentro de la hospitalización (1 = diagnóstico principal) |
| 4 | `icd9_code` | Código de diagnóstico según la Clasificación Internacional de Enfermedades, 9.ª revisión (ICD-9-CM) |

**Variable de análisis principal (col. 4):** `icd9_code` — código ICD-9 del diagnóstico clínico asignado

> **Nota:** La columna `row_id` es un índice interno del sistema y no aporta información clínica, por lo que se excluye del análisis.

In [2]:
dt = pd.read_csv('../Database/PP2_DIAGNOSES_ICD.csv')
# Limpiar nombres de columnas (quitar espacios al inicio/final)
dt.columns = dt.columns.str.strip()

# Excluir columna índice que no aporta valor clínico
columnas_a_excluir = ['row_id']
dt = dt.drop(columns=columnas_a_excluir)

dt.info()
print()
print(dt.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1761 entries, 0 to 1760
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   subject_id  1761 non-null   int64 
 1   hadm_id     1761 non-null   int64 
 2   seq_num     1761 non-null   int64 
 3   icd9_code   1761 non-null   object
dtypes: int64(3), object(1)
memory usage: 55.2+ KB

   subject_id  hadm_id  seq_num icd9_code
0       10006   142345        1     99591
1       10006   142345        2     99662
2       10006   142345        3      5672
3       10006   142345        4     40391
4       10006   142345        5     42731
5       10006   142345        6      4280
6       10006   142345        7      4241
7       10006   142345        8      4240
8       10006   142345        9      2874
9       10006   142345       10     03819


In [3]:
numeric_cols = dt.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = dt.select_dtypes(exclude=[np.number]).columns.tolist()
print('Columnas numéricas:  ', numeric_cols)
print('Columnas categóricas:', categorical_cols)

Columnas numéricas:   ['subject_id', 'hadm_id', 'seq_num']
Columnas categóricas: ['icd9_code']


## 3. Verificación y Limpieza de Datos

Se revisan tres posibles problemas en el dataset:

| Problema | Columna afectada | Acción |
|----------|-----------------|--------|
| NaN reales | Todas | Verificar y reportar |
| Ceros anómalos en IDs | `subject_id`, `hadm_id`, `seq_num` | Verificar (no deben existir) |
| Espacios vacíos o formato incorrecto | `icd9_code` | Normalizar (strip + zfill) |

In [4]:
# --- 3a. Verificar NaN, ceros y espacios vacíos ---
print('=== Valores nulos (NaN) ===')
print(dt.isnull().sum())

print()
print('=== Ceros por columna numérica ===')
for col in numeric_cols:
    zeros = (dt[col] == 0).sum()
    estado = '⚠ ATENCIÓN' if zeros > 0 else '✓ OK'
    print(f'  {col}: {zeros} ceros  {estado}')

print()
print('=== Espacios vacíos en columnas de texto ===')
for col in categorical_cols:
    blancos = (dt[col].astype(str).str.strip() == '').sum()
    estado = '⚠ ATENCIÓN' if blancos > 0 else '✓ OK'
    print(f'  {col}: {blancos} blancos  {estado}')

=== Valores nulos (NaN) ===
subject_id    0
hadm_id       0
seq_num       0
icd9_code     0
dtype: int64

=== Ceros por columna numérica ===
  subject_id: 0 ceros  ✓ OK
  hadm_id: 0 ceros  ✓ OK
  seq_num: 0 ceros  ✓ OK

=== Espacios vacíos en columnas de texto ===
  icd9_code: 0 blancos  ✓ OK


In [5]:
# --- 3b. Normalizar icd9_code ---
# Los códigos ICD-9 numéricos deben tener mínimo 3 dígitos (ej: '039' no '39')
# Los códigos V y E son alfanuméricos y se mantienen tal cual

def normalizar_icd9(codigo):
    codigo = str(codigo).strip().upper()
    if codigo.isdigit():
        codigo = codigo.zfill(3)  # rellena con ceros a la izquierda hasta 3 dígitos
    return codigo

dt['icd9_code'] = dt['icd9_code'].apply(normalizar_icd9)

print('Distribución de longitud de códigos ICD-9 tras normalización:')
print(dt['icd9_code'].str.len().value_counts().sort_index())
print()
print(f'Códigos únicos: {dt["icd9_code"].nunique()}')
print('Muestra de códigos normalizados:')
print(dt['icd9_code'].unique()[:15])

Distribución de longitud de códigos ICD-9 tras normalización:
icd9_code
3     81
4    864
5    816
Name: count, dtype: int64

Códigos únicos: 581
Muestra de códigos normalizados:
['99591' '99662' '5672' '40391' '42731' '4280' '4241' '4240' '2874'
 '03819' '7850' 'E8791' 'V090' '56211' '28529']


## 4. Codificación: texto → entero

`icd9_code` es la única columna de texto. Se convierte a entero con `LabelEncoder` para que el dataset sea completamente numérico y listo para entrenar modelos.

El encoder asigna un número entero a cada código ICD-9 único (en orden alfabético). Se guarda el encoder para poder decodificar después si es necesario.

In [6]:
le_icd9 = LabelEncoder()
dt['icd9_code'] = le_icd9.fit_transform(dt['icd9_code'])

print(f'Clases en icd9_code: {len(le_icd9.classes_)} códigos únicos')
print()
print('Ejemplos de mapeo (código original → entero):')
for i, clase in enumerate(le_icd9.classes_[:10]):
    print(f'  {clase} → {i}')
print('  ...')
print()
print('Para decodificar de vuelta usa: le_icd9.inverse_transform([0, 1, 2])')
print('Ejemplo:', le_icd9.inverse_transform([0, 1, 2]))

Clases en icd9_code: 581 códigos únicos

Ejemplos de mapeo (código original → entero):
  00845 → 0
  0380 → 1
  03811 → 2
  03812 → 3
  03819 → 4
  0383 → 5
  03840 → 6
  03842 → 7
  03843 → 8
  03849 → 9
  ...

Para decodificar de vuelta usa: le_icd9.inverse_transform([0, 1, 2])
Ejemplo: ['00845' '0380' '03811']


## 5. Verificación Final

Confirmación de que el dataset está completamente limpio: sin `NaN`, sin espacios vacíos, sin ceros anómalos, y todas las columnas en formato numérico entero.

In [7]:
print('=== VERIFICACIÓN FINAL ===')
print(f'Forma del dataset:        {dt.shape}')
print(f'¿Quedan NaN?              {dt.isnull().sum().sum()} nulos')
print(f'¿Quedan espacios vacíos?  {(dt.astype(str).apply(lambda c: c.str.strip() == "")).sum().sum()}')
print()
print('Tipos de datos finales:')
print(dt.dtypes)
print()
print('Estadísticas descriptivas:')
print(dt.describe())
print()
print('Primeras 5 filas del dataset limpio:')
print(dt.head())

=== VERIFICACIÓN FINAL ===
Forma del dataset:        (1761, 4)
¿Quedan NaN?              0 nulos
¿Quedan espacios vacíos?  0

Tipos de datos finales:
subject_id    int64
hadm_id       int64
seq_num       int64
icd9_code     int64
dtype: object

Estadísticas descriptivas:
         subject_id        hadm_id      seq_num    icd9_code
count   1761.000000    1761.000000  1761.000000  1761.000000
mean   30392.649631  152515.138558     8.844974   263.382169
std    15461.173204   27552.374513     6.315866   158.225274
min    10006.000000  100375.000000     1.000000     0.000000
25%    10094.000000  128293.000000     4.000000   122.000000
50%    40687.000000  155297.000000     8.000000   237.000000
75%    42199.000000  174863.000000    12.000000   373.000000
max    44228.000000  199395.000000    37.000000   580.000000

Primeras 5 filas del dataset limpio:
   subject_id  hadm_id  seq_num  icd9_code
0       10006   142345        1        473
1       10006   142345        2        477
2       1000